# Public-source note

S5e motif CWM plotting notebook. Outputs stripped; paths sanitized.

In [ ]:
suppressPackageStartupMessages({
  library(dplyr)
  library(ggplot2)
  library(data.table)
  library(GenomicRanges)
  library(BiocParallel)
  library(ComplexHeatmap)
  library(patchwork)
  library(reticulate)
  library(rtracklayer)
  library(tidyr)
    
library(circlize)
library(dplyr)
library(ggplot2)
library(data.table)
library(GenomicRanges)
library(BiocParallel)
library(ComplexHeatmap)
})

In [ ]:
figure_dir_old <- "/path/to/CEBP_analysis/data/ChromBPNet_Fig4/figures_atac_chromhmm/"

figure_dir <- "/path/to/CEBP_analysis/data/ChromBPNet_Fig4_V2/"
finmo_CWMs_dir <-"/path/to/CEBPB_seq/chrombpnet_hits/Combined_modisco_1000_5000000_all_CEBPB/pos_patterns/pos_patterns/merged/CWMs/"
tomtom_cwm_dir <- "/path/to/CEBPB_seq/chrombpnet_hits/Combined_modisco_1000_5000000_all_CEBPB/pos_patterns/pos_patterns/tomtom_cwm/"
modisco_html_path <- "/path/to/CEBPB_seq/chrombpnet_hits/Combined_modisco_1000_5000000_all_CEBPB/pos_patterns/pos_patterns/merged_modisco.html/motifs.html"



In [ ]:
motifs_meme <- universalmotif::read_meme("/path/to/external_analysis/HumanReprogrammingMultiome/Iris/multiome_V2/tables/ChromBPNet_full//motifs.meme.txt")

In [ ]:
pattren_names_df <- read.table(
  paste(figure_dir_old,"/pattren_names.csv",sep=""),
  header = TRUE,
  sep = ",",
    row.names = 1,
  stringsAsFactors = FALSE
)

In [ ]:
head(pattren_names_df)

In [ ]:
pattren_names_df <- read.table(
  paste("5e_EP_motifs_min_prop_0.01_with_KC_names.csv",sep=""),
  header = TRUE,
  sep = ",",
    row.names = 1,
  stringsAsFactors = FALSE
)


In [ ]:
head(pattren_names_df)

In [ ]:
ordered_patterns <- pattren_names_df$pattern
# ordered_patterns <- head(pattren_names_df,10)$pattern
# ordered_matched_motif <- manual_ordered_df$matched_motif
# ordered_pattern_num <- manual_ordered_df$pattern_num
# ordered_pattern_Orientation <- manual_ordered_df$Orientation
# ordered_pattern_Selected_TF <- manual_ordered_df$Selected_TF


In [ ]:
ordered_patterns

In [ ]:
motif_list_modisco <- list()

for (i in seq_along(ordered_patterns)) {
  pattern_sel   <- ordered_patterns[i]
    

  motif_modisco_fc <- read.table(paste0(finmo_CWMs_dir, pattern_sel, "/modisco_fc.txt"))
  rownames(motif_modisco_fc) <- c("A","C","G","T")
  colnames(motif_modisco_fc) <- 1:ncol(motif_modisco_fc)
  motif_list_modisco[[paste("Modisco_FC", pattern_sel)]] <- as.matrix(motif_modisco_fc)

}

In [ ]:
tomtom_cwm_dir

In [ ]:
length(motif_list_modisco)

In [ ]:

figure_file <-   "Motifs_logs_modisco_Selective_Enh_state_KC_ordered.pdf"

# helper once, outside the loop
logo_panel <- function(m, title) {
  if (is.null(m)) {
    ggplot() + theme_void() + ggtitle(paste(title, "(missing)"))
  } else {
    ggplot() +
      ggseqlogo::geom_logo(list(m), method = "custom") +
      ggseqlogo::theme_logo() +
      ggtitle(title) +
      theme(
        plot.title = element_text(size = 8, hjust = 0.5),
        axis.title.x = element_blank(),
        axis.title.y = element_blank(),
        axis.text.x  = element_blank(),
        axis.text.y  = element_blank(),
        axis.ticks.x = element_blank(),
        axis.ticks.y = element_blank(),
        panel.grid   = element_blank(),
        panel.border = element_blank(),
        plot.margin  = margin(0, 0, 0, 0)
      )
  }
}

get_tf_from_key <- function(k) {
  if (is.na(k) || length(k) == 0) return("missing")
  parts <- strsplit(k, " ")[[1]]
  if (length(parts) >= 3) parts[3] else "missing"
}

rows <- vector("list", length(ordered_patterns))

for (i in seq_along(ordered_patterns)) {
    pattern_sel <- ordered_patterns[i]
    # matched_motif <- ordered_matched_motif_updated[i]
    
      key_fc     <- paste("Modisco_FC", pattern_sel)
    
    fc     <- motif_list_modisco[[key_fc]]
    # known1 <-  motif_list_known[[paste(matched_motif, pattern_sel)]]
    
      pat_idx <- which(ordered_patterns == pattern_sel)
    
    
      p1 <- logo_panel(fc,    paste(ordered_patterns[i]))
      # p2 <- logo_panel(known1, matched_motif)
    
     rows[[i]] <- (p1 ) +
        plot_annotation(title = pattern_sel) &
        theme(
          plot.title = element_text(size = 9, hjust = 0),
          plot.margin = margin(0, 0, 0, 0)
        )
}

# stack all rows vertically into one figure
fig_all <- wrap_plots(rows, ncol = 1)

# set page height based on number of motifs
n <- length(rows)
# pdf(figure_file, width = 10, height = 30, useDingbats = FALSE)
# pdf(figure_file, width = 10, height = 80, useDingbats = FALSE)
pdf(figure_file, width = 6, height = 80, useDingbats = FALSE)
print(fig_all)
dev.off()



In [ ]:
length(motif_list_modisco)